In [0]:
import os
import shutil
import pandas as pd
import pyarrow as pa

# show all columns when printing DataFrames
pd.set_option("display.max_columns", None)

In [0]:


SCHEMA = pa.schema([
    ("flight_number", pa.string()),
    ("airline", pa.string()),
    ("origin", pa.string()),
    ("destination", pa.string()),
    ("search_route", pa.string()),
    ("departure_time", pa.string()),
    ("arrival_time", pa.string()),
    ("duration_minutes", pa.int64()),
    ("stops", pa.int64()),
    ("price", pa.float64()),
    ("currency", pa.string()),
    ("trip_type", pa.string()),
    ("fetched_at", pa.string()),
])

In [0]:
def read_bronze(table_name):
    # Read a Bronze Delta table into a DataFrame
    try:
        df = spark.read.table(table_name)
        pdf = df.toPandas()
        print(f"[silver] {table_name} loaded: {len(pdf)} rows")
        return pdf
    except Exception as e:
        print(f"[silver] {table_name} not found — run Bronze script first")
        return pd.DataFrame()

In [0]:
outbound = read_bronze("workspace.bronze.airflights_outbound")
returns = read_bronze("workspace.bronze.airflights_return")
df=clean(outbound)

# Check if data was loaded
if not outbound.empty:
    display(outbound.head())
if not returns.empty:
    display(returns.head())

In [0]:
def clean(df):
    # Add search_route column and convert duration string to total minutes
    df["search_route"] = df["origin"] + " → " + df["destination"]
    df["duration_minutes"] = pd.to_numeric(df["duration"], errors="coerce").fillna(0).astype(int)
    df = df.drop(columns=["duration"])
    df = df.drop_duplicates(subset=["flight_number", "departure_time", "trip_type"])
    df = df[[
        "flight_number", "airline", "origin", "destination", "search_route",
        "departure_time", "arrival_time", "duration_minutes", "stops",
        "price", "currency", "trip_type", "fetched_at"
    ]]
    print(f"[silver] cleaned rows: {len(df)}")
    return df

In [0]:
from deltalake import write_deltalake

def write_silver(df, table_name):
    # Write cleaned DataFrame to Unity Catalog table
    if df.empty:
        print(f"[silver] no data for {table_name} — skipping write")
        return
    
    # Write as Spark DataFrame to Unity Catalog
    spark_df = spark.createDataFrame(df)
    spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print(f"[silver] {len(df)} rows written to {table_name}")


# Use correct table names from Cell 4
outbound = read_bronze("workspace.bronze.airflights_outbound")
returns = read_bronze("workspace.bronze.airflights_return")

combined = pd.concat([outbound, returns], ignore_index=True)
cleaned = clean(combined)
direct = cleaned[cleaned["stops"] == 0]
connecting = cleaned[cleaned["stops"] > 0]

print(f"[silver] direct flights: {len(direct)} | connecting flights: {len(connecting)}")

write_silver(direct, "workspace.silver.airflights_direct")
write_silver(connecting, "workspace.silver.airflights_connecting")

In [0]:
%sql
select* from workspace.silver.airflights_connecting